In [ ]:
#| default_exp reports_interactive

In [ ]:
#| export
import panel as pn

In [ ]:
#| export
from portfolio.plots import timeseries_plot
from portfolio.portfolio import *

In [ ]:
from portfolio.sample_data import *

In [ ]:
#| export
def time_period_w():
    years = list(range(1982, 2026))
    months = list(range(1, 13))
    start_year_w = pn.widgets.Select(name='Start Year', options=years, value=1982)
    start_month_w = pn.widgets.Select(name='Start Month', options=months, value=1)
    end_year_w = pn.widgets.Select(name='End Year', options=years, value=2025)
    end_month_w = pn.widgets.Select(name='End Month', options=months, value=1)
    return start_year_w, start_month_w, end_year_w, end_month_w

In [ ]:
#| export
def interactive_report(*portfolios):
    single = len(portfolios) == 1
    window_w = pn.widgets.Select(name='Rolling Window (Years)', options=list(range(1,30)), value=1)
    time_w = time_period_w()

    slice_ports = lambda sy, sm, ey, em: [p.between(f'{sm}/{sy}', f'{em}/{ey}') for p in portfolios]
    ports_rx = pn.bind(slice_ports, *time_w)

    summary       = pn.bind(lambda ps: pn.panel(compare(*ps)), ports_rx)
    cum_ret_plot  = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='cum_excess_return')), ports_rx)
    roll_ret_plot = pn.bind(lambda ps, w: timeseries_plot(compare(*ps, metric='roll_return', months=w*12), hline=0), ports_rx, window_w)
    real_w_plot   = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='real_w'), logy=True, is_perc=False), ports_rx)
    drawdown_plot = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='drawdown_series')), ports_rx)

    overview = pn.Column(
        pn.pane.Markdown('## Time Period'), pn.Row(*time_w),
        pn.pane.Markdown('## Portfolio Stats'),
        pn.pane.Markdown('### Summary'), summary,
        pn.pane.Markdown('## Portfolio Returns'),
        pn.pane.Markdown('### Cumulative Excess Return'), cum_ret_plot,
        pn.pane.Markdown('### Rolling Excess Return'), window_w, roll_ret_plot,
        pn.pane.Markdown('### Real Wealth (Total Real Cum. Compounded Return)'), real_w_plot,
        pn.pane.Markdown('## Drawdowns'), drawdown_plot)

    if single:
        asset_stats   = pn.bind(lambda ps: pn.panel(ps[0].asset_stats()), ports_rx)
        risk_contrib  = pn.bind(lambda ps: pn.panel(ps[0].risk_contribution()), ports_rx)
        roll_vol_plot = pn.bind(lambda ps: timeseries_plot(ps[0].rolling_vol()), ports_rx)
        rc_plot       = pn.bind(lambda ps: timeseries_plot(ps[0].rc_simple()), ports_rx)
        roll_no_exc   = pn.bind(lambda ps: timeseries_plot(ps[0].roll_return(excess=False, extras=True), hline=0), ports_rx)
        correlation   = pn.bind(lambda ps: pn.panel(ps[0].correlation()), ports_rx)
        overview.extend([
            pn.pane.Markdown('### Asset Stats'), asset_stats,
            pn.pane.Markdown('## Risk'),
            pn.pane.Markdown('### Risk Contribution'), risk_contrib,
            pn.pane.Markdown('### Rolling Volatility'), roll_vol_plot,
            pn.pane.Markdown('### Risk Contribution (Plot)'), rc_plot,
            pn.pane.Markdown('### Extra 12m Rolling (No Excess)'), roll_no_exc])
        corr = pn.Column(pn.pane.Markdown('### Correlation Matrix'), correlation)
        return pn.Tabs(('Portfolio Overview', overview), ('Correlation Matrix', corr))
    return pn.Tabs(('Portfolio Overview', overview))

To create a report you simply pass it one or more portfolios

In [ ]:
rets, rf, cpi = sample_data()

In [ ]:
p = Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi)
p

{'bonds': 0.4, 'stocks': 0.6}

In [ ]:
#| eval: false
pn.panel(interactive_report(p)).save('report.html')